# Differential RV recovery - order-chunk CCF + packaged ccfopt

Minimal E2E check of RV bias with the **packaged** static
`*.ccfopt.delta.mas` (see `examples/rebuild_ccfopt_multi_bulk_differential.py`).
This notebook does **not** rebuild or overwrite that mask.

Stellar RV uses the **default** `ccf_line_mask` chunking path:
`window_line_by_line=False` and `window_size_um=0.25` (equal-um wavelength
chunks along each order, then inverse-variance aggregation) -- not the
single-line `CCFOPT_SINGLE_LINE_STELLAR_RV_KW` mode.

**Runtime (estimate):** about **30-90 minutes** for the default 7-bulk grid
(`BULK_RV_KM_S` below: 14 full densify/UTR/Stage0-2 injections plus one
shared Stage-1 bundle). Order-chunk CCF is much cheaper than line-by-line;
wall time is still machine-dependent and scales roughly with
`len(BULK_RV_KM_S)`. Use a shorter list for a quick check.

> For each **bulk** in `BULK_RV_KM_S`, inject `(bulk, bulk + drv)` with
> `drv = 5 m/s`. Report absolute biases and the **matched-chunk** differential
> with joint inliers + hard QC. Require
> `|estimated_drv - drv| < 10 cm/s` at every bulk.

Physical path (`inject_rv_via='spectrum'`, fiber **2**):

1. Densify BT-Settl -> Doppler paint -> near-noiseless UTR -> Stage 0.
2. Stage 1: box extract (aperture 8) + shared Aidan WAVE + deg-0 order ZP;
   shared fiberflat / traces from the RV=0 bundle.
3. Stage 2: blaze; **barycorr skipped**; stellar RV with default order-chunk
   `ccf_line_mask` kwargs and `packaged_default_line_mask_path()`.

Sequence:

1. Build shared Stage-1 bundle at RV=0.
2. For each bulk in `BULK_RV_KM_S` (default `-150...+150` km/s), inject
   `bulk` and `bulk + 5 m/s`.
3. Score matched-chunk `|estimated_drv - drv|` against 10 cm/s per bulk.

**Environment variables** (set before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache |
| `KECK_OBSERVER_EMAIL` | Optional remote sync | Keck Observer email |
| `KECK_OBSERVER_PASSWORD` | Optional remote sync | Keck Observer password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.


In [ ]:
import os

import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# Order-chunk products live under OUT; keep KOA cache shared with sibling runs.
OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim/order_chunk"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    "/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE",
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)

Uses `ensure_koa_calibrations`: seed the notebook cache from another populated
KOA cache if needed, try remote sync, and fall back to local-only on auth /
network / API failures. After that, this notebook pins
`HISPEC_KOA_CONNECT_REMOTE=0` so Stage 0-2 lookups stay offline.


In [ ]:
from hispecdrp.calibrations import ensure_koa_calibrations

SPECTROGRAPH = "BSPEC"

# Optional: pin an explicit seed cache (otherwise auto-discovers
# ~/hispec*/cals/KOA_CALIBRATION_CACHE and HISPEC_KOA_CACHE_SEED).
# KOA_SEED_SOURCES = ["/Users/ckilpatrick/hispec_rv_order_scan/cals/KOA_CALIBRATION_CACHE"]
KOA_SEED_SOURCES = None

summary = ensure_koa_calibrations(
    SPECTROGRAPH,
    seed_sources=KOA_SEED_SOURCES,
    ignore_missing_cal_types=["lfc_wavesol"],
)

# Keep the rest of this notebook offline: Stage 0/1 primitives must not
# re-login to KOA when the remote API/auth is down.
os.environ["HISPEC_KOA_CONNECT_REMOTE"] = "0"

seed = summary.get("seed") or {}
if seed.get("seeded"):
    print(f"seeded KOA cache from {seed['source']}  ({seed['n_fits_before']} -> {seed['n_fits_after']} FITS)")
elif seed.get("n_fits_before", 0):
    print(f"KOA cache already populated ({seed['n_fits_before']} FITS)")
print(f"remote_mode: {summary.get('remote_mode')}")
if summary.get("remote_error"):
    print(f"remote_error: {summary['remote_error']}")
print("HISPEC_KOA_CONNECT_REMOTE:", os.environ.get("HISPEC_KOA_CONNECT_REMOTE"))

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])
if summary.get("still_missing"):
    print("still_missing:", summary["still_missing"])


## 2. Injection configuration

Fiber 2. Fixed Aidan WAVE + deg-0 order ZP. Spectrum Doppler paint only.
Near-noiseless UTR. Uses the packaged static `ccfopt.delta.mas` (**not**
rebuilt here) with **default order-chunk** stellar RV
(`window_line_by_line=False`, `window_size_um=0.25`).

Edit **`BULK_RV_KM_S`** below to choose bulk offsets. Default sweeps
`-150...+150` km/s in 50 km/s steps. For each bulk, injects are
`(bulk)` and `(bulk + 5 m/s)`.


In [ ]:
from pathlib import Path

from hispecdrp.utils.line_mask_utils import (
    MaskLines,
    packaged_default_line_mask_path,
    read_mask_generation_meta,
)

FIBERS = [2]
INJECT_RV_VIA = "spectrum"  # densify -> Doppler paint; WAVE fixed (Aidan + ZP0)
EXTRACTION_APERTURE = 8
TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
# Packaged default mask (same as resolve_mask_path(None)); do not overwrite here.
MASK_PATH = packaged_default_line_mask_path()

POISSON_NOISE = False
CONVERT_TO_UINT16 = False

SEED = 0
DRV_M_S = 5.0
REQUIREMENT_M_S = 0.10  # 10 cm/s

# Bulk RV offsets (km/s). Edit this list to run a subset or different grid.
# For each bulk B, injects are B and B + DRV_M_S (m/s).
BULK_RV_KM_S = [-150.0, -100.0, -50.0, 0.0, 50.0, 100.0, 150.0]


def bulk_run_label(bulk_km_s: float, which: str) -> str:
    """Stable run-dir label for a bulk / epoch (a=bulk, b=bulk+drv)."""
    sign = "m" if bulk_km_s < 0 else "p"
    return f"bulk_{sign}{abs(bulk_km_s):06.1f}kms_{which}".replace(".", "p")


def stellar_rv_prim_args(ccf_input_velocity_m_s: float, *, save_preview: bool = False) -> dict:
    """Default order-chunk ccf_line_mask (StellarRVPrimitive defaults).

    Explicitly keeps window_line_by_line=False and window_size_um=0.25.
    Does not use CCFOPT_SINGLE_LINE_STELLAR_RV_KW.
    """
    return {
        "algorithm": "ccf_line_mask",
        "mask_path": MASK_PATH,
        "ccf_input_velocity": float(ccf_input_velocity_m_s),
        "do_fibers": list(FIBERS),
        # Default chunking: equal-um bins along each order, then aggregate.
        "window_line_by_line": False,
        "window_size_um": 0.25,
        # Sim: BERV forced 0; keep CCF zb identically 0.
        "ccf_zb": 0.0,
        "ccf_zb_from_barycorr": False,
        "save_preview": bool(save_preview),
        "preview_write_png": False,
        "show_preview": False,
    }


assert Path(MASK_PATH).is_file(), f"missing packaged mask: {MASK_PATH}"
_mask_meta = read_mask_generation_meta(MASK_PATH) or {}
_n_mask = len(MaskLines.load(MASK_PATH).line_lambda_um)
_n_inj = 2 * len(BULK_RV_KM_S)
_srv = stellar_rv_prim_args(0.0)
print(f"fiber(s): {FIBERS}")
print(f"extraction_aperture = {EXTRACTION_APERTURE}")
print(f"inject_rv_via = {INJECT_RV_VIA!r}  (densify then Doppler only at ZEMAX)")
print(f"poisson_noise={POISSON_NOISE}  convert_to_uint16={CONVERT_TO_UINT16}")
print(f"mask_path = {MASK_PATH}")
print(
    f"mask n_lines={_n_mask}  "
    f"generator={_mask_meta.get('generator')}"
)
print(
    f"stellar RV: order-chunk  "
    f"window_line_by_line={_srv['window_line_by_line']}  "
    f"window_size_um={_srv['window_size_um']}"
)
print(f"seed={SEED}  drv={DRV_M_S:.3f} m/s")
print(f"BULK_RV_KM_S = {BULK_RV_KM_S}")
print(
    f"per bulk: inject bulk and bulk+{DRV_M_S:.3f} m/s  "
    f"({_n_inj} injections total)"
)
print(
    f"runtime estimate: ~30-90 min for {_n_inj} full E2E injections "
    f"(scales roughly with len(BULK_RV_KM_S)={len(BULK_RV_KM_S)})"
)
print(f"requirement: |estimated_drv - drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")


## 3. Shared Stage-1 cals (fiberflat / traces / Aidan+ZP0 WAVE) + injection helper

Build once at RV=0, then reuse fiberflat / traces / WAVE for every injection.
Stage-2 skips barycorr; stellar RV uses default order-chunk `ccf_line_mask`
via `stellar_rv_prim_args`.


In [ ]:
from pathlib import Path

from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
from hispecdrp.simulations.e2e_pipeline import build_wavesol_aidan_with_order_zp
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

# --- Shared Stage-1 bundle (fiberflat + traces + Aidan WAVE + deg-0 order ZP) ---
_shared_dir = str(Path(OUT) / "_shared_stage1_rv0")
os.makedirs(_shared_dir, exist_ok=True)
print(
    "Building shared Stage-1 cals (RV=0 -> fiberflat/traces aperture="
    f"{EXTRACTION_APERTURE}, Aidan gws + order ZP0) ..."
)
_ramp0, _cals0, RATE_MAP_REF = create_ramp_from_spectrum(
    _shared_dir,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename=TEMPLATE,
    mag_ab=3.05,
    filter_band="J",
    seed=SEED,
    radial_velocity_km_s=0.0,
    barycentric_velocity_km_s=0.0,
    inject_rv_via=INJECT_RV_VIA,
    poisson_noise=POISSON_NOISE,
    convert_to_uint16=CONVERT_TO_UINT16,
    extraction_aperture=float(EXTRACTION_APERTURE),
    reuse_stage1_sims=False,
)
SHARED_TRACEPOS = _cals0["trace_pos"]
SHARED_TRACEPROF = _cals0["trace_prof"]
SHARED_FIBERFLAT = _cals0["fiberflat"]
SHARED_WAVESOL, ZP_INFO = build_wavesol_aidan_with_order_zp(
    RATE_MAP_REF,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    known_rv_m_s=0.0,
    meta={"mjd_start": getattr(getattr(_ramp0, "meta", None), "mjd_start", 0.0)},
)
print(
    f"SHARED_WAVESOL note={getattr(SHARED_WAVESOL.meta, 'note', None)}  "
    f"median order ZP = {ZP_INFO.get('median_dv_m_s', float('nan')):.3f} m/s  "
    f"(poly_deg={ZP_INFO.get('poly_deg')})"
)
print(
    f"shared fiberflat/traces: "
    f"ff={SHARED_FIBERFLAT is not None}  "
    f"tracepos={SHARED_TRACEPOS is not None}  "
    f"traceprof={SHARED_TRACEPROF is not None}"
)


def run_one_injection(
    label: str,
    injected_rv_km_s: float,
    seed: int,
    *,
    save_preview: bool = False,
) -> dict:
    """Densify->Doppler->ZEMAX UTR, Stage0-2 with shared cals; return aggregate RV (m/s)."""
    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    ramp, _cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        register_stage1_sims=False,  # use shared fiberflat / traces / WAVE
    )
    print(f"ramp={ramp.meta.filename}  shape={ramp.data.shape}")

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=run_dir).apply(ramp)
    l1 = Stage1Pipeline(
        output_dir=run_dir,
        prim_args={
            "spec_extract": {
                "do_fibers": do_fibers,
                "method": "box",
                "extraction_aperture": int(EXTRACTION_APERTURE),
                "tracepos": SHARED_TRACEPOS,
                "traceprof": SHARED_TRACEPROF,
            },
            "set_wavesol": {"wavesol": SHARED_WAVESOL},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=run_dir,
        prim_args={
            "blaze_corr": {
                "do_fibers": do_fibers,
                "fiberflat": SHARED_FIBERFLAT,
            },
            # BERV forced 0 at sim time; skip barycorr so CCF zb stays identically 0.
            "barycorr": {"run": False},
            "stellar_rv": stellar_rv_prim_args(
                injected_m_s, save_preview=bool(save_preview)
            ),
        },
    ).apply(l1)

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is None or len(avg) == 0:
        raise RuntimeError(f"{label}: missing rvccfavg_fib{fib}")
    rv_rec = float(avg["RV"][0])
    rv_err = float(avg["RV_ERR"][0])
    preview_html = None
    if save_preview:
        preview_html = l2.get_output_path(
            output_dir=run_dir, suffix="stellar_rv"
        ).replace(".fits", "_preview.html")
        print(f"stellar RV preview HTML: {preview_html}")
    print(
        f"fiber {fib}: recovered RV = {rv_rec:.9f} +/- {rv_err:.9f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.9f} m/s)"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "wavesol": SHARED_WAVESOL,
        "preview_html": preview_html,
        "l2": l2,
    }


## 4. Inject each bulk pair (order-chunk stellar RV; no mask rebuild)

For every entry in `BULK_RV_KM_S`, run injects **A** = bulk and
**B** = bulk + 5 m/s on the shared densify->Doppler path / shared cals.
Reports `bias = recover - inject` at each inject.


In [ ]:
import numpy as np

RESULTS = {}  # bulk_km_s -> {"a": result, "b": result}
drv_km_s = DRV_M_S / 1000.0

for i_bulk, bulk_km_s in enumerate(BULK_RV_KM_S):
    bulk = float(bulk_km_s)
    lab_a = bulk_run_label(bulk, "a")
    lab_b = bulk_run_label(bulk, "b")
    save_preview = i_bulk == 0
    result_a = run_one_injection(lab_a, bulk, SEED, save_preview=save_preview)
    result_b = run_one_injection(
        lab_b, bulk + drv_km_s, SEED, save_preview=False
    )
    RESULTS[bulk] = {"a": result_a, "b": result_b}

# Sanity: WAVE arrays should match across injections (fixed WAVE path)
_wave_attr = f"wave_fib{FIBERS[0]}"
_first = RESULTS[float(BULK_RV_KM_S[0])]
w_a = np.asarray(getattr(_first["a"]["wavesol"], _wave_attr), dtype=np.float64)
w_b = np.asarray(getattr(_first["b"]["wavesol"], _wave_attr), dtype=np.float64)
print(
    f"\nmax |WAVE fib{FIBERS[0]} (B - A)| = {np.nanmax(np.abs(w_b - w_a)):.3e}  "
    f"(should be ~0 for inject_rv_via='spectrum')"
)

print("\n=== Absolute recovery / bias (order-chunk stellar RV) ===")
for bulk in BULK_RV_KM_S:
    for which in ("a", "b"):
        r = RESULTS[float(bulk)][which]
        print(
            f"{r['label']:28s} inject = {r['inject_m_s']:12.3f} m/s  "
            f"recover = {r['recover_m_s']:.9f} +/- {r['recover_err_m_s']:.9f} m/s  "
            f"bias = {r['recover_m_s'] - r['inject_m_s']:.9f} m/s"
        )


## 5. Differential score vs 10 cm/s (matched-chunk, per bulk)

Uses `estimate_matched_line_differential_rv` on the per-chunk `rvccf` table
(joint inliers + hard QC). Require `|estimated_drv - drv| < 10 cm/s` at
**every** bulk in `BULK_RV_KM_S`.


In [ ]:
from hispecdrp.primitives.stellar_rv.rv_differential_utils import (
    estimate_matched_line_differential_rv,
)


def _score_pair_matched(name: str, a: dict, b: dict) -> dict:
    """Score drv with matched-chunk delta + joint inliers."""
    drv = b["inject_m_s"] - a["inject_m_s"]
    fib = FIBERS[0]
    ta = getattr(a["l2"], f"rvccf_fib{fib}")
    tb = getattr(b["l2"], f"rvccf_fib{fib}")
    diff = estimate_matched_line_differential_rv(
        ta,
        tb,
        inject_a_m_s=float(a["inject_m_s"]),
        inject_b_m_s=float(b["inject_m_s"]),
    )
    estimated_drv = float(diff["estimated_drv_m_s"])
    residual = float(diff["residual_m_s"])
    passed = bool(np.isfinite(residual) and abs(residual) < REQUIREMENT_M_S)

    print(f"=== Differential {name} (matched-chunk) ===")
    print(
        f"  A inject = {a['inject_m_s']:.9f} m/s  recover = {a['recover_m_s']:.9f}  "
        f"bias = {a['recover_m_s'] - a['inject_m_s']:.9f} m/s"
    )
    print(
        f"  B inject = {b['inject_m_s']:.9f} m/s  recover = {b['recover_m_s']:.9f}  "
        f"bias = {b['recover_m_s'] - b['inject_m_s']:.9f} m/s"
    )
    print(f"  drv                 = {drv:.12f} m/s")
    print(
        f"  estimated_drv (matched-chunk) = {estimated_drv:.12f} +/- "
        f"{diff['estimated_drv_err_m_s']:.12f} m/s  "
        f"(n_kept={diff['n_kept']}/{diff['n_matched']})"
    )
    print(
        f"  estimated_drv - drv = {residual:.12f} m/s  "
        f"({residual * 100.0:.9f} cm/s)"
    )
    print(
        f"  requirement         = |estimated_drv - drv| < {REQUIREMENT_M_S:.2f} m/s "
        f"-> {'PASS' if passed else 'FAIL'}"
    )
    if not passed:
        print(
            f"  WARNING: residual {residual:.12f} m/s exceeds {REQUIREMENT_M_S:.2f} m/s"
        )
    return {
        "name": name,
        "bulk_km_s": float(a["inject_km_s"]),
        "drv": drv,
        "estimated_drv": estimated_drv,
        "estimated_drv_err": float(diff["estimated_drv_err_m_s"]),
        "residual": residual,
        "residual_cm_s": residual * 100.0,
        "passed": passed,
        "n_kept": int(diff["n_kept"]),
        "n_matched": int(diff["n_matched"]),
    }


SCORES = []
for bulk in BULK_RV_KM_S:
    pair = RESULTS[float(bulk)]
    name = f"bulk={bulk:+.1f} km/s (drv={DRV_M_S:.0f} m/s)"
    SCORES.append(_score_pair_matched(name, pair["a"], pair["b"]))
    print()


def _fmt_bulk_km_s(bulk: float) -> str:
    if abs(float(bulk)) < 0.5:
        return "0"
    return f"{float(bulk):+.0f}"


def _fmt_resid_cm_s(resid: float) -> str:
    return f"{float(resid):+.2f}"


import pandas as pd
from IPython.display import display

summary = pd.DataFrame(
    [
        {
            "bulk (km/s)": _fmt_bulk_km_s(s["bulk_km_s"]),
            "residual (cm/s)": _fmt_resid_cm_s(s["residual_cm_s"]),
            "n_chunks_kept": f"{s['n_kept']}/{s['n_matched']}",
            "result": "PASS" if s["passed"] else "FAIL",
        }
        for s in SCORES
    ]
)
display(summary)

n_pass = sum(1 for s in SCORES if s["passed"])
print(f"Overall: {n_pass}/{len(SCORES)} PASS")
assert n_pass == len(SCORES), (
    f"{len(SCORES) - n_pass} bulk(s) failed |estimated_drv - drv| < {REQUIREMENT_M_S} m/s"
)
